# Bloomberg Data Extraction: G10 Bond Futures & US Equity Option Chains

This notebook builds on the FX-futures example in `Bloomberg_Data_Extraction.ipynb`
and pulls two much larger datasets via the official Bloomberg API (`blpapi`):

1. **G10 bond futures** — every tenor we can find, using *generic* tickers
   (`TY1 Comdty`, `RX1 Comdty`, …) so we get the **longest continuous history
   available**. Per the *Bloomberg Open API – Core User Guide* (§3.5.11),
   *"When requesting historical data, generics string together contracts based
   on the generics' settings and go back in history as long as data is
   available (in contrast, individual contract Tickers would only have
   historical data up to its first trading date)."* That is exactly the
   property we want.

2. **Historical option chains on US equities** — for a configurable list of
   underlyings (SPY, QQQ, AAPL, …) we
   - pull the live option chain via the bulk field `OPT_CHAIN`
     (*Core User Guide* §3.5.12.1 — `OPT_CHAIN = BDS(Underlying, "OPT_CHAIN")`),
   - then download end-of-day price, bid/ask, volume, open interest and
     implied vol for every contract via `HistoricalDataRequest`.

   ⚠️ *Core User Guide* §3.5.12 (p.27) is explicit: *"Option Tickers are
   kept up to three years after expiry"*. "Comprehensive" therefore means
   everything Bloomberg still has on disk — typically ~3 years of expired
   contracts plus all currently listed ones. There is no API-side workaround
   for that retention window.

### Prerequisites

- A running Bloomberg Terminal on the **same machine** (logged in).
- `blpapi` installed from Bloomberg's index:
  ```bash
  pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi
  ```
- `pandas`, `numpy`, and optionally `pyarrow` (for Parquet output).

### Output

All raw data is written under `./data/` in a structure documented at the end.
CSV by default so the files open anywhere; Parquet is an opt-in for the
larger options dataset.

## 1. Configuration

All tunable knobs are in this cell. The defaults are conservative — short
date range for options, modest underlying list — so a first run completes
in a sensible amount of time. Expand them for production pulls.

In [ ]:
# ---- Date range ----------------------------------------------------------
# Bond futures: use a very early start date — the generic tickers will return
# data from whenever Bloomberg's series actually begins (often the 1980s for
# US Treasuries, 1990s for Bund/Gilt, etc.).
BOND_START_DATE = "19800101"
BOND_END_DATE   = None          # None -> today

# Options: Bloomberg retains expired option tickers ~3y, so pulling from ~5y
# ago covers everything still on disk plus all currently listed contracts.
OPT_START_DATE  = "20200101"
OPT_END_DATE    = None

# ---- Universe of US-equity underlyings for options ----------------------
# Edit freely. ETFs, indices, single names — anything with listed options.
OPTION_UNDERLYINGS = [
    "SPY US Equity",     # S&P 500 ETF
    "QQQ US Equity",     # Nasdaq-100 ETF
    "IWM US Equity",     # Russell 2000 ETF
    "AAPL US Equity",
    "MSFT US Equity",
    "NVDA US Equity",
    "AMZN US Equity",
    "GOOGL US Equity",
    "TSLA US Equity",
    "META US Equity",
]

# Cap how many contracts per underlying we pull history for. SPY alone can
# have 10k+ option tickers; every one of them costs an API hit. None -> pull
# all of them (comprehensive but slow / quota-heavy).
MAX_OPTIONS_PER_UNDERLYING = 500

# ---- Output --------------------------------------------------------------
OUTPUT_DIR        = "../data"
USE_PARQUET_OPTS  = False        # True if pyarrow/fastparquet is installed
                                 # (~5x smaller than CSV for option history)

# ---- Batch sizes ---------------------------------------------------------
# Core Developer Guide notes the API auto-splits into groups of ~10
# securities x 128 fields. Keep batches modest to avoid timeouts and huge
# in-memory events.
HIST_BATCH_SIZE  = 50    # securities per HistoricalDataRequest
REFDATA_BATCH    = 100   # securities per ReferenceDataRequest

import datetime as dt
if BOND_END_DATE is None:
    BOND_END_DATE = dt.datetime.today().strftime("%Y%m%d")
if OPT_END_DATE is None:
    OPT_END_DATE = dt.datetime.today().strftime("%Y%m%d")

print(f"Bond futures window : {BOND_START_DATE} -> {BOND_END_DATE}")
print(f"Options window      : {OPT_START_DATE} -> {OPT_END_DATE}")
print(f"Underlyings         : {len(OPTION_UNDERLYINGS)} tickers")
print(f"Output directory    : ./{OUTPUT_DIR}/")


## 2. Open the Bloomberg session

Same pattern as the FX notebook — connect to `localhost:8194` and open the
`//blp/refdata` service. *Core Developer Guide* §13 lays out the canonical
Request/Response flow we'll use everywhere below:

1. `Session.start()`
2. `Session.openService("//blp/refdata")`
3. `service.createRequest("HistoricalDataRequest" | "ReferenceDataRequest")`
4. populate `securities` / `fields` elements (and overrides if needed)
5. `session.sendRequest(request)`
6. loop on `session.nextEvent()` until we see `Event.RESPONSE`, processing
   each `PARTIAL_RESPONSE` along the way (large pulls always arrive in
   chunks).

In [ ]:
import os
import time
from pathlib import Path

import blpapi
import pandas as pd
import numpy as np

# Make output directories upfront so saving never fails mid-run.
ROOT = Path(OUTPUT_DIR)
(ROOT / "bond_futures").mkdir(parents=True, exist_ok=True)
(ROOT / "equity_options").mkdir(parents=True, exist_ok=True)

def open_session():
    """Open a synchronous Bloomberg session on localhost:8194."""
    opts = blpapi.SessionOptions()
    opts.setServerHost("localhost")
    opts.setServerPort(8194)
    s = blpapi.Session(opts)
    if not s.start():
        raise RuntimeError("Failed to start Bloomberg session - is the Terminal running?")
    if not s.openService("//blp/refdata"):
        raise RuntimeError("Failed to open //blp/refdata service")
    return s

session = open_session()
refdata = session.getService("//blp/refdata")
print("Bloomberg session open.")


## 3. Generic request helpers

Three small functions cover everything we need. The shape of each response
is documented in the *Core Developer Guide*:
- §5.4.2 — reference-data response (the array-of-securityData pattern)
- §13.2 — historical-data response (a *single* `securityData` per message)
- §5.4.3 — bulk reference-data response (each field row is itself a
  sequence, traversed via `numValues()` then `numElements()`)

Our wrappers below return tidy `pandas` structures so downstream code never
has to touch the Bloomberg `Element` machinery directly.

In [ ]:
def _drain_response(session, parse_msg):
    """Loop on session.nextEvent() until a RESPONSE event arrives. For each
    Message in every PARTIAL_RESPONSE / RESPONSE event call parse_msg(msg, rows)
    where rows is a list that parse_msg can append to. Returns the final rows."""
    rows = []
    while True:
        ev = session.nextEvent(500)
        ev_type = ev.eventType()
        if ev_type in (blpapi.Event.PARTIAL_RESPONSE, blpapi.Event.RESPONSE):
            for msg in ev:
                # Top-level responseError -> request failed (bad date, no auth, ...)
                if msg.hasElement("responseError"):
                    err = msg.getElement("responseError")
                    raise RuntimeError(f"Bloomberg responseError: {err}")
                parse_msg(msg, rows)
        if ev_type == blpapi.Event.RESPONSE:
            break
    return rows


def historical_request(session, securities, fields,
                       start_date, end_date,
                       periodicity="DAILY",
                       non_trading_day_fill="NON_TRADING_WEEKDAYS",
                       currency=None,
                       adjustment_normal=True,
                       adjustment_abnormal=True,
                       adjustment_split=True):
    """Fetch HistoricalDataRequest data for many securities x many fields.

    Returns a long DataFrame with columns [date, ticker, field, value].
    Securities are auto-chunked into HIST_BATCH_SIZE-sized requests.
    """
    svc = session.getService("//blp/refdata")
    all_rows = []
    securities = list(securities)
    fields     = list(fields)

    n_batches = (len(securities) - 1) // HIST_BATCH_SIZE + 1
    for i in range(0, len(securities), HIST_BATCH_SIZE):
        batch = securities[i:i + HIST_BATCH_SIZE]
        req = svc.createRequest("HistoricalDataRequest")
        for s in batch:
            req.getElement("securities").appendValue(s)
        for f in fields:
            req.getElement("fields").appendValue(f)
        req.set("startDate", start_date)
        req.set("endDate",   end_date)
        req.set("periodicitySelection",   periodicity)
        req.set("nonTradingDayFillOption", non_trading_day_fill)
        req.set("adjustmentNormal",   adjustment_normal)
        req.set("adjustmentAbnormal", adjustment_abnormal)
        req.set("adjustmentSplit",    adjustment_split)
        if currency:
            req.set("currency", currency)

        session.sendRequest(req)

        def _parse(msg, rows):
            if not msg.hasElement("securityData"):
                return
            sd = msg.getElement("securityData")
            # HistoricalDataResponse: securityData is a single element per msg.
            ticker = sd.getElementAsString("security")
            if sd.hasElement("securityError"):
                print(f"  WARN securityError for {ticker}: "
                      f"{sd.getElement('securityError')}")
                return
            fd = sd.getElement("fieldData")
            for j in range(fd.numValues()):
                row = fd.getValue(j)
                d   = row.getElementAsDatetime("date")
                for f in fields:
                    if row.hasElement(f):
                        try:
                            v = row.getElementAsFloat(f)
                        except Exception:
                            v = row.getElementAsString(f)
                        rows.append((d, ticker, f, v))

        all_rows.extend(_drain_response(session, _parse))
        print(f"  hist batch {i // HIST_BATCH_SIZE + 1}/{n_batches} "
              f"({len(batch)} securities, "
              f"cumulative rows: {len(all_rows):,})")

    df = pd.DataFrame(all_rows, columns=["date", "ticker", "field", "value"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
    return df


def reference_request(session, securities, fields, overrides=None):
    """Static (scalar) reference data — returns DataFrame indexed by ticker."""
    svc = session.getService("//blp/refdata")
    out_rows = []
    securities = list(securities)
    fields     = list(fields)

    for i in range(0, len(securities), REFDATA_BATCH):
        batch = securities[i:i + REFDATA_BATCH]
        req = svc.createRequest("ReferenceDataRequest")
        for s in batch:
            req.getElement("securities").appendValue(s)
        for f in fields:
            req.getElement("fields").appendValue(f)
        if overrides:
            ovs = req.getElement("overrides")
            for k, v in overrides.items():
                ov = ovs.appendElement()
                ov.setElement("fieldId", k)
                ov.setElement("value",   str(v))

        session.sendRequest(req)

        def _parse(msg, rows):
            if not msg.hasElement("securityData"):
                return
            sd_arr = msg.getElement("securityData")
            for k in range(sd_arr.numValues()):
                sd = sd_arr.getValueAsElement(k)
                ticker = sd.getElementAsString("security")
                rec = {"ticker": ticker}
                if sd.hasElement("securityError"):
                    rec["securityError"] = str(sd.getElement("securityError"))
                if sd.hasElement("fieldData"):
                    fd = sd.getElement("fieldData")
                    for f in fields:
                        if fd.hasElement(f):
                            try:
                                rec[f] = fd.getElementAsFloat(f)
                            except Exception:
                                rec[f] = fd.getElementAsString(f)
                rows.append(rec)

        out_rows.extend(_drain_response(session, _parse))

    if not out_rows:
        return pd.DataFrame()
    return pd.DataFrame(out_rows).set_index("ticker")


def bulk_request(session, security, field, overrides=None):
    """Pull a bulk (array-valued) field — e.g. OPT_CHAIN, FUT_CHAIN — for a
    single security. Returns a list of dicts, one per row in the bulk array.

    Refs: Core Developer Guide §5.4.3 (bulk message handling),
          Core User Guide §3.5.12.1 (OPT_CHAIN syntax).
    """
    svc = session.getService("//blp/refdata")
    req = svc.createRequest("ReferenceDataRequest")
    req.getElement("securities").appendValue(security)
    req.getElement("fields").appendValue(field)
    if overrides:
        ovs = req.getElement("overrides")
        for k, v in overrides.items():
            ov = ovs.appendElement()
            ov.setElement("fieldId", k)
            ov.setElement("value", str(v))
    session.sendRequest(req)

    def _parse(msg, rows):
        if not msg.hasElement("securityData"):
            return
        sd_arr = msg.getElement("securityData")
        for k in range(sd_arr.numValues()):
            sd = sd_arr.getValueAsElement(k)
            if sd.hasElement("securityError"):
                print(f"  WARN {security}: {sd.getElement('securityError')}")
                continue
            if not sd.hasElement("fieldData"):
                continue
            fd = sd.getElement("fieldData")
            if not fd.hasElement(field):
                continue
            bulk = fd.getElement(field)
            # Each bulk-array row is itself a sub-sequence.
            for b in range(bulk.numValues()):
                sub = bulk.getValueAsElement(b)
                rec = {}
                for n in range(sub.numElements()):
                    el = sub.getElement(n)
                    try:
                        rec[str(el.name())] = el.getValueAsString()
                    except Exception:
                        rec[str(el.name())] = None
                rows.append(rec)

    return _drain_response(session, _parse)

print("Helpers ready: historical_request, reference_request, bulk_request")


## 4. Part A — G10 bond futures universe

### What's a "G10 bond futures" universe?

G10 currencies are USD, EUR, JPY, GBP, CHF, CAD, AUD, NZD, NOK, SEK. Of
those, **liquid exchange-listed government-bond futures only exist for
seven**: US, Germany (the EUR proxy), Japan, UK, Canada, Australia, and
Switzerland. Norway, Sweden, and New Zealand do not have liquid
exchange-traded bond futures, so they are omitted (a 3y NZ contract exists
but is illiquid and history is short).

The table below uses the Bloomberg short codes you'd see on `CT <GO>` /
`CTM <GO>`. The *Core User Guide* §3.5.11.1 defines the ticker format as
`Root x Month x Year x Session x Yellow Key`; the *generic* form replaces
`Month x Year` with an ordinal `1`, `2`, …  (User Guide p.23,
*Generic Ticker* row). So `TY1 Comdty` is the front-month 10y T-note,
`TY2 Comdty` is the following expiry, and so on. Generics are what give us
the *longest* continuous history.

| Country     | Tenor        | Root | Exchange | Generic ticker |
|-------------|--------------|------|----------|----------------|
| US          | 2y           | TU   | CBT      | `TU1 Comdty`   |
| US          | 3y           | 3Y   | CBT      | `3Y1 Comdty`   |
| US          | 5y           | FV   | CBT      | `FV1 Comdty`   |
| US          | 10y          | TY   | CBT      | `TY1 Comdty`   |
| US          | Ultra 10y    | UXY  | CBT      | `UXY1 Comdty`  |
| US          | Long Bond    | US   | CBT      | `US1 Comdty`   |
| US          | Ultra Bond   | WN   | CBT      | `WN1 Comdty`   |
| Germany     | Schatz 2y    | DU   | Eurex    | `DU1 Comdty`   |
| Germany     | Bobl 5y      | OE   | Eurex    | `OE1 Comdty`   |
| Germany     | Bund 10y     | RX   | Eurex    | `RX1 Comdty`   |
| Germany     | Buxl 30y     | UB   | Eurex    | `UB1 Comdty`   |
| UK          | Short Gilt   | WB   | ICE      | `WB1 Comdty`   |
| UK          | Long Gilt    | G    | ICE      | `G 1 Comdty`   |
| Japan       | 10y JGB      | JB   | OSE      | `JB1 Comdty`   |
| Japan       | JGB Mini     | JM   | OSE      | `JM1 Comdty`   |
| Canada      | 2y CGZ       | CV   | MX       | `CV1 Comdty`   |
| Canada      | 10y CGB      | CN   | MX       | `CN1 Comdty`   |
| Canada      | 30y LGB      | LB   | MX       | `LB1 Comdty`   |
| Australia   | 3y           | YM   | ASX      | `YM1 Comdty`   |
| Australia   | 10y          | XM   | ASX      | `XM1 Comdty`   |
| Australia   | 20y          | YXT  | ASX      | `YXT1 Comdty`  |
| Switzerland | CONF 10y     | SWG  | Eurex    | `SWG1 Comdty`  |

**Note on single-letter roots:** the UK Long Gilt root is just `G`, so the
ticker needs a space — `G 1 Comdty`. The *Core User Guide* p.23 shows this
explicitly (`G H20 Comdty` for a specific contract). Our helper handles the
spacing automatically.

We also pull the second generic (`TU2`, `FV2`, …) so users get the front
plus back month — enough for any roll / calendar-spread analysis. Bump
`GENERIC_DEPTH` to get more of the forward curve.

In [ ]:
# (country, tenor, root, exchange) — keep this as data, easy to extend.
BOND_FUTURES_SPEC = [
    ("US",          "2y",          "TU",  "CBT"),
    ("US",          "3y",          "3Y",  "CBT"),
    ("US",          "5y",          "FV",  "CBT"),
    ("US",          "10y",         "TY",  "CBT"),
    ("US",          "Ultra 10y",   "UXY", "CBT"),
    ("US",          "Long Bond",   "US",  "CBT"),
    ("US",          "Ultra Bond",  "WN",  "CBT"),
    ("Germany",     "Schatz 2y",   "DU",  "Eurex"),
    ("Germany",     "Bobl 5y",     "OE",  "Eurex"),
    ("Germany",     "Bund 10y",    "RX",  "Eurex"),
    ("Germany",     "Buxl 30y",    "UB",  "Eurex"),
    ("UK",          "Short Gilt",  "WB",  "ICE"),
    ("UK",          "Long Gilt",   "G",   "ICE"),   # single letter -> space
    ("Japan",       "JGB 10y",     "JB",  "OSE"),
    ("Japan",       "JGB Mini",    "JM",  "OSE"),
    ("Canada",      "CGZ 2y",      "CV",  "MX"),
    ("Canada",      "CGB 10y",     "CN",  "MX"),
    ("Canada",      "LGB 30y",     "LB",  "MX"),
    ("Australia",   "3y",          "YM",  "ASX"),
    ("Australia",   "10y",         "XM",  "ASX"),
    ("Australia",   "20y",         "YXT", "ASX"),
    ("Switzerland", "CONF 10y",    "SWG", "Eurex"),
]

# Pull the first two generic contracts -> front + back month. Increase to
# get more of the forward curve (more API quota usage though).
GENERIC_DEPTH = 2

def generic_ticker(root: str, n: int) -> str:
    """Bloomberg generic-future ticker. Single-letter roots take a space
    (Core User Guide §3.5.11.1 - `G H20 Comdty` shows the same convention)."""
    sep = " " if len(root) == 1 else ""
    return f"{root}{sep}{n} Comdty"

bond_tickers = []
bond_meta    = []
for country, tenor, root, exch in BOND_FUTURES_SPEC:
    for n in range(1, GENERIC_DEPTH + 1):
        tkr = generic_ticker(root, n)
        bond_tickers.append(tkr)
        bond_meta.append(dict(ticker=tkr, country=country, tenor=tenor,
                              root=root, exchange=exch, generic_n=n))

bond_meta_df = pd.DataFrame(bond_meta)
print(f"Built {len(bond_tickers)} bond-futures tickers "
      f"({len(BOND_FUTURES_SPEC)} contracts x {GENERIC_DEPTH} generics):")
bond_meta_df.head(10)


### 4.1 Download bond-futures history

We ask for `PX_LAST`, `PX_OPEN`, `PX_HIGH`, `PX_LOW`, `VOLUME`, `OPEN_INT`.
The earliest available date for each series is determined by Bloomberg —
US Treasury futures often go back to the 1980s, Bund/Gilt to the early
1990s, JGB and Canadian/Australian contracts somewhat later.

Bloomberg's daily-limit caveats from the FX notebook still apply — if a
call returns no data, switch Terminals or wait 24h. *Core User Guide* §9.2.5
documents the `DAILY_LIMIT_REACHED` / `MONTHLY_LIMIT_REACHED` errors.

In [ ]:
BOND_FIELDS = [
    "PX_LAST", "PX_OPEN", "PX_HIGH", "PX_LOW",
    "VOLUME",  "OPEN_INT",
]

print(f"Requesting {len(bond_tickers)} tickers x {len(BOND_FIELDS)} fields "
      f"from {BOND_START_DATE} to {BOND_END_DATE}...")
t0 = time.time()

bond_long = historical_request(
    session,
    securities=bond_tickers,
    fields=BOND_FIELDS,
    start_date=BOND_START_DATE,
    end_date=BOND_END_DATE,
)

print(f"\nDone in {time.time() - t0:.1f}s. Total rows: {len(bond_long):,}")
bond_long.head()


### 4.2 Reshape and save

Three complementary views of the same data:

1. **Long format** (`bond_futures_long.csv`) — one row per
   `(date, ticker, field)`. Best for tidy-data tooling.
2. **Wide per field** (`bond_futures_px_last.csv`, …) — date index × ticker
   columns. Same shape as the FX notebook's output, so existing analysis
   code drops straight in.
3. **Metadata** (`bond_futures_metadata.csv`) — country, tenor, exchange,
   root, generic-month index for every ticker.

In [ ]:
bond_dir = ROOT / "bond_futures"

# 1) Long format ----------------------------------------------------------
bond_long.to_csv(bond_dir / "bond_futures_long.csv", index=False)
print(f"  wrote {bond_dir/'bond_futures_long.csv'}   "
      f"({len(bond_long):,} rows)")

# 2) Wide format, one CSV per field --------------------------------------
for fld in BOND_FIELDS:
    sub = bond_long[bond_long["field"] == fld]
    if sub.empty:
        print(f"  (no data for {fld}, skipping)")
        continue
    wide = (sub.pivot_table(index="date", columns="ticker", values="value")
               .sort_index())
    fname = bond_dir / f"bond_futures_{fld.lower()}.csv"
    wide.to_csv(fname)
    print(f"  wrote {fname}   shape={wide.shape}")

# 3) Metadata ------------------------------------------------------------
bond_meta_df.to_csv(bond_dir / "bond_futures_metadata.csv", index=False)
print(f"  wrote {bond_dir/'bond_futures_metadata.csv'}   ({len(bond_meta_df)} tickers)")

# Quick sanity check
px_path = bond_dir / "bond_futures_px_last.csv"
if px_path.exists():
    px = pd.read_csv(px_path, index_col=0, parse_dates=True)
    print(f"\nPX_LAST table: {px.shape[0]:,} dates x {px.shape[1]} tickers")
    print(f"Date range    : {px.index.min().date()} -> {px.index.max().date()}")


## 5. Part B — US equity option chains

For each underlying:

1. Get the live chain with `BDS(underlying, "OPT_CHAIN")`. This is a *bulk*
   field — *Core User Guide* §3.5.12.1 documents the syntax; *Core
   Developer Guide* §5.4.3 walks through the bulk-response shape (each row
   is a sub-sequence with one element per sub-field — for `OPT_CHAIN` that
   sub-element is named *"Security Description"*).
2. *Optionally* cap the chain at `MAX_OPTIONS_PER_UNDERLYING`, prioritising
   currently listed contracts.
3. Pull static metadata (strike, expiry, put/call, exercise style…) in one
   `ReferenceDataRequest`.
4. Pull daily history (price, bid/ask, volume, open interest, IV) via
   `HistoricalDataRequest`, auto-batched.

### 5.1 Fetch the option chains

In [ ]:
# OPT_CHAIN bulk rows historically come through as "Security Description";
# some Bloomberg builds return alternative casings — we normalise.
def _extract_ticker_from_chain_row(row):
    for k in ("Security Description", "security_description",
              "Security_Description", "SECURITY_DESCRIPTION"):
        if k in row and row[k]:
            return row[k]
    vals = [v for v in row.values() if v]
    return vals[0] if vals else None


option_chains = {}        # underlying  ->  list[str]   (option tickers)
chain_summary = []

for underlying in OPTION_UNDERLYINGS:
    print(f"\nFetching option chain for {underlying} ...")
    rows = bulk_request(session, underlying, "OPT_CHAIN")
    tickers = [t for t in (_extract_ticker_from_chain_row(r) for r in rows) if t]
    # De-duplicate while preserving order
    seen = set()
    tickers = [t for t in tickers if not (t in seen or seen.add(t))]
    option_chains[underlying] = tickers
    chain_summary.append(dict(underlying=underlying, n_contracts=len(tickers)))
    print(f"  -> {len(tickers):,} option tickers in chain")

chain_summary_df = pd.DataFrame(chain_summary)
chain_summary_df


### 5.2 Pull static option metadata

For every option ticker we collect: strike, expiry, put/call flag, exercise
style, contract size, multiplier, and the underlying — enough to slice the
dataset however you like (by expiry bucket, moneyness, OTM/ITM, …) without
going back to Bloomberg.

These are all standard reference fields; pull more up on `FLDS <GO>` if you
need (e.g. `OPT_DELTA_MID`, `OPT_GAMMA_MID`, `OPT_IMPLIED_VOLATILITY_MID`).

In [ ]:
OPT_META_FIELDS = [
    "OPT_STRIKE_PX",
    "OPT_EXPIRE_DT",
    "OPT_PUT_CALL",
    "OPT_EXERCISE_TYP",
    "OPT_CONT_SIZE",
    "OPT_UNDL_TICKER",
    "OPT_MULTIPLIER",
]

all_meta = {}        # underlying -> metadata DataFrame

for underlying, tickers in option_chains.items():
    if not tickers:
        continue
    print(f"\nMetadata: {underlying}  ({len(tickers):,} contracts)")
    # Cap before pulling metadata to save quota
    use = tickers[:MAX_OPTIONS_PER_UNDERLYING] if MAX_OPTIONS_PER_UNDERLYING else tickers

    meta = reference_request(session, use, OPT_META_FIELDS)
    if not meta.empty:
        if "OPT_EXPIRE_DT" in meta.columns:
            meta["OPT_EXPIRE_DT"] = pd.to_datetime(meta["OPT_EXPIRE_DT"],
                                                   errors="coerce")
        if "OPT_STRIKE_PX" in meta.columns:
            meta["OPT_STRIKE_PX"] = pd.to_numeric(meta["OPT_STRIKE_PX"],
                                                  errors="coerce")
    all_meta[underlying] = meta
    print(f"  -> metadata for {len(meta):,} contracts")
    if not meta.empty:
        print(meta.head(3))


### 5.3 Trim chains to the most useful contracts

If `MAX_OPTIONS_PER_UNDERLYING` is set we keep:
- all currently *listed* (non-expired) contracts, expiry-ascending, then
- expired contracts closest to today, until we hit the cap.

Set `MAX_OPTIONS_PER_UNDERLYING = None` in cell 1 to pull every contract
Bloomberg still has — *that* is the literal "comprehensive" history
available. Be warned: a SPY pull can exceed 10,000 contracts × ~1,000
trading days × 6 fields = 60 M cells.

In [ ]:
def select_contracts(meta, max_n, today=None):
    """Return the (up to) max_n most-useful option tickers from a metadata table."""
    if meta is None or meta.empty:
        return pd.Index([])
    if not max_n or len(meta) <= max_n:
        return meta.index

    if today is None:
        today = pd.Timestamp.today().normalize()

    m = meta.copy()
    if "OPT_EXPIRE_DT" in m.columns:
        m["_expired"]   = m["OPT_EXPIRE_DT"] < today
        m["_dist_days"] = (m["OPT_EXPIRE_DT"] - today).abs().dt.days.fillna(1e9)
    else:
        m["_expired"]   = False
        m["_dist_days"] = 0

    live    = m[~m["_expired"]].sort_values("OPT_EXPIRE_DT").index
    expired = m[m["_expired"]].sort_values("_dist_days").index

    keep = list(live[:max_n])
    if len(keep) < max_n:
        keep += list(expired[: max_n - len(keep)])
    return pd.Index(keep)


contracts_to_pull = {}     # underlying -> list[str]
for u, meta in all_meta.items():
    keep = select_contracts(meta, MAX_OPTIONS_PER_UNDERLYING)
    contracts_to_pull[u] = list(keep)
    print(f"{u:<18s}  pulling {len(keep):>5,} / {len(option_chains[u]):>5,} contracts")


### 5.4 Download historical data for every selected option

Fields:

- `PX_LAST` — last traded price (often missing on thinly-traded strikes)
- `PX_BID`, `PX_ASK` — end-of-day quotes
- `VOLUME` — contracts traded that day
- `OPEN_INT` — open interest at close
- `IVOL_LAST` — last implied volatility (Bloomberg's surface model)

The *Core User Guide* (§3.5.12, p.27) is explicit that the historical
retention for option tickers is bounded (~3y post-expiry), so older tickers
will silently return no data — that's expected.

We save **per-underlying** so the run is restartable: if a network blip
kills the kernel after AAPL is finished but before MSFT, AAPL's file is
already on disk and you can re-run from where you stopped.

In [ ]:
OPT_HIST_FIELDS = [
    "PX_LAST", "PX_BID", "PX_ASK",
    "VOLUME", "OPEN_INT",
    "IVOL_LAST",
]

def save_options_history(underlying, df_long):
    """Persist one underlying's option history. Parquet if enabled & importable,
    else gzipped CSV (option history is huge but very compressible)."""
    safe = underlying.replace(" ", "_").replace("/", "_")
    out_dir = ROOT / "equity_options" / safe
    out_dir.mkdir(parents=True, exist_ok=True)

    if USE_PARQUET_OPTS:
        try:
            df_long.to_parquet(out_dir / "history.parquet", index=False)
            print(f"  saved {out_dir/'history.parquet'}   "
                  f"({len(df_long):,} rows)")
            return
        except Exception as e:
            print(f"  parquet failed ({e}) - falling back to gzipped CSV")

    fp = out_dir / "history.csv.gz"
    df_long.to_csv(fp, index=False, compression="gzip")
    print(f"  saved {fp}   ({len(df_long):,} rows)")


for underlying, tickers in contracts_to_pull.items():
    if not tickers:
        print(f"\n{underlying}: empty chain, skipping")
        continue
    print(f"\n=== {underlying}: pulling history for {len(tickers):,} contracts ===")
    t0 = time.time()

    df = historical_request(
        session,
        securities=tickers,
        fields=OPT_HIST_FIELDS,
        start_date=OPT_START_DATE,
        end_date=OPT_END_DATE,
    )

    elapsed = time.time() - t0
    print(f"  -> {len(df):,} rows in {elapsed:.1f}s")

    if not df.empty:
        # Attach key metadata so the file is self-describing.
        meta = all_meta.get(underlying)
        if meta is not None and not meta.empty:
            df = df.merge(
                meta[["OPT_STRIKE_PX", "OPT_EXPIRE_DT", "OPT_PUT_CALL"]]
                    .reset_index().rename(columns={"index": "ticker"}),
                on="ticker", how="left")
        df["underlying"] = underlying

    save_options_history(underlying, df)

    # Save the per-contract metadata alongside.
    safe = underlying.replace(" ", "_").replace("/", "_")
    meta_fp = ROOT / "equity_options" / safe / "metadata.csv"
    if underlying in all_meta and not all_meta[underlying].empty:
        all_meta[underlying].to_csv(meta_fp)
        print(f"  saved {meta_fp}")


### 5.5 Save chain summary and close the session

In [ ]:
summary_rows = []
for u in OPTION_UNDERLYINGS:
    chain  = option_chains.get(u, [])
    pulled = contracts_to_pull.get(u, [])
    summary_rows.append(dict(
        underlying    = u,
        chain_size    = len(chain),
        pulled_size   = len(pulled),
        coverage_pct  = round(100 * len(pulled) / max(len(chain), 1), 1),
    ))

summary = pd.DataFrame(summary_rows)
summary.to_csv(ROOT / "equity_options" / "chain_summary.csv", index=False)
summary


In [ ]:
session.stop()
print("Bloomberg session stopped.")


## 6. Output structure

```
data/
├── bond_futures/
│   ├── bond_futures_metadata.csv          ← country / tenor / exchange / generic_n
│   ├── bond_futures_long.csv              ← tidy: date, ticker, field, value
│   ├── bond_futures_px_last.csv           ← wide: date × ticker
│   ├── bond_futures_px_open.csv
│   ├── bond_futures_px_high.csv
│   ├── bond_futures_px_low.csv
│   ├── bond_futures_volume.csv
│   └── bond_futures_open_int.csv
└── equity_options/
    ├── chain_summary.csv                  ← one row per underlying
    ├── SPY_US_Equity/
    │   ├── history.csv.gz (or .parquet)   ← long format, strike/expiry merged in
    │   └── metadata.csv                   ← per-contract static fields
    ├── QQQ_US_Equity/
    │   └── …
    └── …
```

## 7. Notes, gotchas, and how to extend

- **Daily / monthly limits.** Academic Bloomberg licences have an
  undocumented daily data-cap per machine. If a large pull returns empty,
  you've probably hit it — switch Terminals or wait 24h. *Core User Guide*
  §9.2.5 lists the `DAILY_LIMIT_REACHED` / `MONTHLY_LIMIT_REACHED` response
  errors the API raises.

- **Why generic vs specific tickers for futures.** A specific ticker like
  `TYZ24 Comdty` only has data from its first trading date forward; the
  generic `TY1 Comdty` chains successive contracts together via the
  Bloomberg rollover rules (`CDEF <GO>` on the Terminal) and gives the
  longest possible series. For the term structure on a particular date,
  request `TY1 … TY8 Comdty` together and pivot by `generic_n` in the
  metadata.

- **Specific contracts.** If you do want individual contract tickers
  (`RXM26 Comdty`, etc.), extend `BOND_FUTURES_SPEC` with explicit month/year
  codes — the Bloomberg month-code table is `FGHJKMNQUVXZ` for Jan–Dec
  (*Core User Guide* §3.5.11.1, p.24).

- **Adjustments.** Bond futures are unaffected by the corporate-action
  flags. The options block sets `adjustmentSplit=True`, etc., which is
  almost always what you want for options on split-adjusted underlyings.

- **Adding more contracts / underlyings.** Edit `BOND_FUTURES_SPEC` or
  `OPTION_UNDERLYINGS` at the top of the notebook and re-run — every
  downstream cell is data-driven.

- **Intraday data.** Replace `HistoricalDataRequest` with
  `IntradayBarRequest` or `IntradayTickRequest` (Developer Guide §13.3 /
  §13.5). Same session, same service, one security per request — no batching.

- **Volatility surface, not just `IVOL_LAST`.** `OVDV <GO>` shows
  per-moneyness / per-delta volatility tickers (e.g. `IBM US APR14 95 VOL
  BVOL Equity` — User Guide §3.5.12, p.27). Pass those through
  `historical_request` exactly like ordinary equities to get continuous
  vol time series without needing to retain expired option tickers.

- **Licensing reminder.** Bloomberg data is licensed per Terminal — files
  produced by this notebook are for use on the machine that produced them,
  not for redistribution.